# E0: smoke test and day-1 gates (G1 preamble, G2 horizon)
Kaggle T4, fp16. Add `HF_TOKEN` as a Kaggle secret (account with gated Tiny Aya access).

In [ ]:
!pip uninstall -y torchao
!pip install -q 'git+https://github.com/dhairya-pandya/Lexsi-AI-Alignment@feat/hsafe' pytest

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

In [ ]:
!git clone -q -b feat/hsafe https://github.com/dhairya-pandya/Lexsi-AI-Alignment repo
%cd repo
!python -m pytest -q -m slow  # real tokenizer: system turn == 359 tokens

In [ ]:
# Coherence check: one P-default and one P-none generation
import torch, yaml
from transformers import AutoTokenizer
from hsafe.gen import HFGenerator
from hsafe.records import PromptRecord
from hsafe.render import render
m = yaml.safe_load(open('configs/models.yaml'))['tiny-aya-global']
tok = AutoTokenizer.from_pretrained(m['id'], revision=m['revision'])
gen = HFGenerator(m['id'], m['revision'], dtype=torch.float16, max_new_tokens=64)
rec = PromptRecord('smoke','smoke','What is your name, and what can you help me with?','en','smoke','rule_probe','NA','name','eval')
for c in ['P-default','P-none']:
    print(c, '->', gen.generate([render(rec, c, None, None, tok, 'tiny_aya')])[0].output)

In [ ]:
del gen; import gc; gc.collect(); torch.cuda.empty_cache()
!python -m hsafe.run --config configs/gate_g1.yaml
!python -m hsafe.run --config configs/gate_g2.yaml

In [ ]:
!python -m hsafe.gates results/gate_g1 results/gate_g2
!cat results/e0/findings.json | head -80

## Decision (design §7)
- G1 pass -> continue as planned.
- G1 fail, G2 pass -> E3-E5 use near (3000) vs far (5500) contrast.
- both fail -> fallback project.
Record the team decision in `results/e0/findings.json['decision']` and commit `results/` (no cache files).